# LLM from Scratch

Building **nanoGPT** inspired by Andrej Kathaparthy's approach

1. It is initially trained on **Tiny Shakespeare** dataset.

In [2]:
#importing libraries
import torch

In [27]:
#constants
torch.manual_seed(42) #overused hitchhikers but still nice

### Downloading the data

- Using `!wget` which downloads the files at a given url.
- URL in this case is https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt.
- The file we download is `input.txt`.

In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-11-13 18:33:42--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2025-11-13 18:33:42 (22.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [4]:
#reading the file
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [5]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
print(len(text))

1115394


### Encoder Decoder

- Encoder is a mapping from character to number.
- Decoder gives the character back based on the number.

This is a simple character level tokenizer.

In [7]:
#creating a sorted list of unique characters that appear and count the total for a vocab size
set_of_text = set(text)
print(set_of_text)
unique_list = list(set_of_text)
print(unique_list)
chars = sorted(unique_list)
print(chars)
vocab_size = len(chars)
print(vocab_size)

{';', ' ', 'E', 'e', '3', 'i', '.', 'p', 'W', 'J', 'A', 'M', 'a', 'N', '!', ',', 'H', 'Z', 'P', '$', 's', 'I', 'K', 'B', 'o', 'Y', 'h', 'r', 'f', 'm', 'l', 'b', ':', 'D', 'u', 'c', 'R', '&', 'w', 'k', 'v', 'L', 'x', 'd', "'", 'z', 'g', 'C', 'O', 'j', 'T', 'F', '\n', '?', 'V', 'Q', 'y', 'S', '-', 'G', 't', 'X', 'U', 'n', 'q'}
[';', ' ', 'E', 'e', '3', 'i', '.', 'p', 'W', 'J', 'A', 'M', 'a', 'N', '!', ',', 'H', 'Z', 'P', '$', 's', 'I', 'K', 'B', 'o', 'Y', 'h', 'r', 'f', 'm', 'l', 'b', ':', 'D', 'u', 'c', 'R', '&', 'w', 'k', 'v', 'L', 'x', 'd', "'", 'z', 'g', 'C', 'O', 'j', 'T', 'F', '\n', '?', 'V', 'Q', 'y', 'S', '-', 'G', 't', 'X', 'U', 'n', 'q']
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [8]:
#or basically
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### Note

Vocab contains both uppercase and lowercase letters with a few special characters in the beginning.

In [9]:
#simple encoder/decoder
#make a dictionary that has character as key and its place as value
#index for the char is found due to enumerate
stoi = {ch:i for i,ch in enumerate(chars)}
#same but other way round
itos = {i:ch for i,ch in enumerate(chars)}
#encoder - for every character in a given string, return a stoi value or index
encode = lambda s:[stoi[c] for c in s]
#decoder
decode = lambda l:''.join([itos[i] for i in l])

In [10]:
def encoder(str):
  arr = []
  for c in str:
    arr.append(stoi[c])
  return arr

In [11]:
def decoder(arr):
  chars = []
  for num in arr:
    chars.append(itos[num])
  return "".join(chars)

In [12]:
print(encode('is this vocab reAdy?'))

[47, 57, 1, 58, 46, 47, 57, 1, 60, 53, 41, 39, 40, 1, 56, 43, 13, 42, 63, 12]


In [13]:
print(decode([2, 4, 5]))

!&'


In [14]:
print(encoder('is this vocab reAdy?'))

[47, 57, 1, 58, 46, 47, 57, 1, 60, 53, 41, 39, 40, 1, 56, 43, 13, 42, 63, 12]


In [15]:
print(decoder([2, 4, 5]))

!&'


### Creating a tensor for encoded text

A pytorch tensor is a multidimensional array like a numpy array with additional powers for autograd or deep learning.

In [16]:
encoded_text = encode(text)

In [17]:
print(text[:50])

First Citizen:
Before we proceed any further, hear


In [18]:
print(encoded_text[:50])

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56]


### Note

Sanity check: `i` of `First` and `Citizen` are both `47`, so the encoder works.

In [19]:
#making a tensor of the data
data = torch.tensor(encoded_text, dtype=torch.long)

In [20]:
print(data[:50])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [21]:
data.shape

torch.Size([1115394])

### Splitting the data

This split is done to keep the test data completely separate.

In [22]:
#training
n = int(0.9*len(data)) #90%
train_data = data[:n]
val_data = data[n:]

### Note

Training the transformer on the entire text would be too computationally expensive. So, only chuncks or blocks of the data are used.

### Why a block size of 8?

The block_size or context_length is the number of previous tokens the model sees when predicting the next one.

Self attention scales as $O(\text{block size})^2$, which will become clear later.

Memory scales as $O(\text{batch size} \times \text{block size})$. So, it'll be 8x4 (4 will later be chosen as the batch size) which is 32.

It's just a small design choice to keep matrices simple for now.

In [23]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

### Explanation

In these 9 characters, 8 examples are packed.

- In a context of 18, 47 comes next.
- In a context of 18 and 47, 56 comes next.

Here's a code snippet used by Andrej Kathaparthy to explain it:

In [25]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [31]:
block_size = 8
batch_size = 4 #process 4 parallel sequences of context length 8

In [38]:
def get_batch(split):
  data = train_data if split=='train' else val_data #go load data based on what we doing

  #torch.randint(low, high, size) or (high, size) in this case
  #generate 4 random locations, where the last one that u can select is len(data)-8
  ix = torch.randint(len(data)- block_size, (batch_size,)) #generate random positions to grab chuncks out of, generate batch_size number of offsets
  #note - (batch_size,) cos it should be tuple of ints, not int
  #example: ix = [4, 3, 9, 7] -> start from indices 4, 3, 9 and 7 to take 8 tokens for input

  #from data, go to i (4, 3, 9 or 7) and take data from i to i+8
  #then, stack them together in a 4x8 tensor
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

In [44]:
xb, yb = get_batch('train')

print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[ 1, 41, 46, 47, 50, 42, 12,  0],
        [59, 57, 39, 52, 42,  1, 57, 58],
        [53, 53, 52,  6,  1, 57, 47, 56],
        [ 1, 58, 46, 59, 52, 42, 43, 56]])
targets:
torch.Size([4, 8])
tensor([[41, 46, 47, 50, 42, 12,  0,  0],
        [57, 39, 52, 42,  1, 57, 58, 56],
        [53, 52,  6,  1, 57, 47, 56, 12],
        [58, 46, 59, 52, 42, 43, 56,  1]])


A transformer processes:

multiple sequences (B)

each sequence over multiple time steps (T)

In [45]:
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

when input is [1] the target: 41
when input is [1, 41] the target: 46
when input is [1, 41, 46] the target: 47
when input is [1, 41, 46, 47] the target: 50
when input is [1, 41, 46, 47, 50] the target: 42
when input is [1, 41, 46, 47, 50, 42] the target: 12
when input is [1, 41, 46, 47, 50, 42, 12] the target: 0
when input is [1, 41, 46, 47, 50, 42, 12, 0] the target: 0
when input is [59] the target: 57
when input is [59, 57] the target: 39
when input is [59, 57, 39] the target: 52
when input is [59, 57, 39, 52] the target: 42
when input is [59, 57, 39, 52, 42] the target: 1
when input is [59, 57, 39, 52, 42, 1] the target: 57
when input is [59, 57, 39, 52, 42, 1, 57] the target: 58
when input is [59, 57, 39, 52, 42, 1, 57, 58] the target: 56
when input is [53] the target: 53
when input is [53, 53] the target: 52
when input is [53, 53, 52] the target: 6
when input is [53, 53, 52, 6] the target: 1
when input is [53, 53, 52, 6, 1] the target: 57
when input is [53, 53, 52, 6, 1, 57] the t